# Hyperparameter Optimization with Snowflake

**Session 3 — Deep Dive: Distributed Training & Hyperparameter Optimization**  
**Notebook 4 of 4** | ~20 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Tuner API** | Snowflake's native distributed hyperparameter optimization |
| **Search Space** | Define parameter ranges with `uniform`, `loguniform`, `randint`, `choice` |
| **Search Algorithms** | RandomSearch, GridSearch, BayesOpt — when to use each |
| **Training Function** | The `train_func` pattern with `get_tuner_context()` |
| **Experiment Tracking** | Each trial logged as an experiment run for comparison |

---

### HPO Architecture on SPCS

```
                  ┌──────────────────────────────────────┐
                  │          SPCS Compute Pool            │
                  │                                      │
  Tuner.run()     │  ┌────────┐  ┌────────┐  ┌────────┐ │
  ───────────>    │  │ Trial 1│  │ Trial 2│  │ Trial 3│ │
                  │  │ lr=0.01│  │ lr=0.05│  │ lr=0.1 │ │
                  │  │ d=6    │  │ d=8    │  │ d=4    │ │
                  │  └───┬────┘  └───┬────┘  └───┬────┘ │
                  │      │           │           │      │
                  │      v           v           v      │
                  │   f1=0.82     f1=0.87     f1=0.79   │
                  │                                      │
                  │  Search Algorithm picks next configs  │
                  │  based on results so far              │
                  └──────────────────────────────────────┘
```

> **Documentation**: [Tune model hyperparameters](https://docs.snowflake.com/en/developer-guide/snowflake-ml/tune-hyperparameters)

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import json
import logging
import os

from utils import get_config
from utils import get_session

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

config = get_config("config.yaml")
DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse
COMPUTE_POOL = config.compute.compute_pool
STAGE = f"{DB}.{SCHEMA}.{config.stages.job_payloads}"

session = get_session(
    connection_name=config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)

print(f"Connected as {session.get_current_user()} | Role: {session.get_current_role()}")
print(f"Compute Pool: {COMPUTE_POOL}")

## 2 | The Snowflake Tuner API

The Tuner API provides a **high-level interface** for distributed HPO.
It handles Ray cluster setup, trial scheduling, and result collection automatically.

### Three Components

1. **Training function** — receives sampled hyperparameters, trains, reports metrics
2. **Search space** — defines parameter ranges to explore
3. **TunerConfig** — controls the search algorithm, metric, and trial count

### Step 1: Define the Search Space

The search space maps parameter names to **sampling functions** that define
the range and distribution of values to explore.

| Function | Distribution | Use For |
|----------|-------------|----------|
| `randint(lo, hi)` | Uniform integer [lo, hi) | `n_estimators`, `max_depth` |
| `uniform(lo, hi)` | Uniform float [lo, hi) | `subsample`, `colsample_bytree` |
| `loguniform(lo, hi)` | Log-uniform float | `learning_rate`, `reg_alpha` |
| `choice([...])` | Categorical | `booster`, `activation` |

In [ ]:
search_space = {
    "n_estimators": {"type": "randint", "range": [50, 300]},
    "max_depth": {"type": "randint", "range": [3, 12]},
    "learning_rate": {"type": "loguniform", "range": [0.001, 0.3]},
    "subsample": {"type": "uniform", "range": [0.6, 1.0]},
    "colsample_bytree": {"type": "uniform", "range": [0.6, 1.0]},
}

print("Search Space (defined in train_hpo_tuner.py):")
for param, spec in search_space.items():
    print(f"  {param:20s} -> {spec['type']}({spec['range'][0]}, {spec['range'][1]})")

### Step 2: The Training Function

The training function runs **once per trial** inside SPCS. It receives sampled
hyperparameters via `get_tuner_context()` and reports metrics back.

Our script `train_hpo_tuner.py` also logs each trial to **Experiment Tracking**
so we can compare all trials in Snowsight after the HPO run.

#### Key Pattern

```python
def train_func():
    ctx = get_tuner_context()
    params = ctx.get_hyper_params()   # Sampled config for this trial
    datasets = ctx.get_dataset_map()  # Data connectors passed to Tuner.run()
    
    # ... train model with params ...
    
    ctx.report(
        metrics={"f1_macro": score},   # Must include TunerConfig.metric
        model=trained_model,            # Optional: saved for best_model access
    )
```

> **Note**: This function runs **server-side** inside SPCS. It has access to
> Snowflake data via `DataConnector`, but not to your local filesystem.

### Step 3: Configure the Tuner

### Search Algorithms Compared

| Algorithm | How It Works | Best When |
|-----------|-------------|----------|
| **RandomSearch** | Uniform random sampling | Large search spaces, exploration phase |
| **GridSearch** | Exhaustive grid over all combos | Small, discrete spaces (< 100 combos) |
| **BayesOpt** | Gaussian process models metric surface | Expensive evaluations, continuous params |

#### Decision Guide

```
 Start here
    │
    ├── Budget < 20 trials? ──> BayesOpt (smart sampling)
    │
    ├── Discrete params only? ──> GridSearch (if combos < 100)
    │
    └── Large continuous space? ──> RandomSearch (parallelize well)
```

In [ ]:
tuner_config = {
    "metric": "f1_macro",
    "mode": "max",
    "search_alg": "RandomSearch",
    "num_trials": 10,
}

print("Tuner Configuration:")
for k, v in tuner_config.items():
    print(f"  {k:15s} {v}")

### Step 4: Submit the HPO Job

We invoke the Tuner API which submits the HPO job to SPCS. It manages the Ray cluster,
runs trials in parallel, and each trial logs itself to Experiment Tracking.

> With 10 trials on a small dataset, expect ~5-10 minutes depending on compute pool capacity.

In [ ]:
from snowflake.ml.jobs import submit_directory

JOB_DIR = "session3_distributed_training_and_hpo/job_payload"
EXPERIMENT_NAME = "PATIENT_RISK_HPO_SESSION3"
NUM_TRIALS = 10

job = submit_directory(
    dir_path=JOB_DIR,
    entrypoint="train_hpo_tuner.py",
    compute_pool=COMPUTE_POOL,
    stage_name=STAGE,
    target_instances=2,
    env_vars={
        "DATABASE": DB,
        "SCHEMA": SCHEMA,
        "EXPERIMENT_NAME": EXPERIMENT_NAME,
        "NUM_TRIALS": NUM_TRIALS,
    },
)

print(f"HPO job submitted!")
print(f"  Job ID: {job.id}")
print(f"  Status: {job.status}")
print(f"  Trials: {NUM_TRIALS}")

### Step 5: Wait for Completion and View Logs

The HPO job runs server-side. Each trial trains a model, reports metrics to the
Tuner, and logs itself to Experiment Tracking.

In [ ]:
print(f"Waiting for HPO job {job.id} ...")
print("(This may take 5-10 minutes)\n")

job.wait()
print(f"Final status: {job.status}")

logs = job.get_logs()
print("\n" + "=" * 60)
print("JOB LOGS (tail)")
print("=" * 60)
print(logs[-3000:] if len(logs) > 3000 else logs)

## 3 | View Trials in Experiment Tracking

Each trial was logged to the `PATIENT_RISK_HPO_SESSION3` experiment by the training function.
You can view these in Snowsight under **AI & ML > Experiments**, or query them here.

In [ ]:
import json

runs = session.sql(f"SHOW RUNS IN EXPERIMENT {EXPERIMENT_NAME}").to_pandas()
print(f"Experiment runs logged: {len(runs)}")
display(runs)

best_run = None
if len(runs) > 0:
    metadata = runs['"metadata"'].apply(json.loads)
    runs["f1_macro"] = metadata.apply(
        lambda m: m.get("metrics", {}).get("f1_macro", {}).get("value")
    )
    runs["accuracy"] = metadata.apply(
        lambda m: m.get("metrics", {}).get("accuracy", {}).get("value")
    )
    runs["run_name"] = runs['"name"']
    valid = runs.dropna(subset=["f1_macro"])
    if len(valid) > 0:
        best_run = valid.sort_values("f1_macro", ascending=False).iloc[0]

if best_run is not None:
    print(f"\nBest trial: {best_run['run_name']}")
    print(f"  f1_macro: {best_run['f1_macro']:.4f}")
    print(f"  accuracy: {best_run['accuracy']:.4f}")

## 4 | Register the Best Model

Since the Tuner ran server-side, we don't have `best_model` locally.
Two options to register:

1. **Add `registry.log_model()` inside `train_hpo_tuner.py`** after `tuner.run()` completes
2. **Re-train locally with the best params** from the experiment runs

Below we take the best hyperparameters from Experiment Tracking and submit
a final training job with those exact params.

In [ ]:
if best_run is not None:
    best_params = {
        "N_ESTIMATORS": str(int(float(best_run.get("n_estimators", 100)))),
        "MAX_DEPTH": str(int(float(best_run.get("max_depth", 8)))),
        "LEARNING_RATE": str(best_run.get("learning_rate", "0.1")),
    }

    job_best = submit_directory(
        dir_path="session3_distributed_training_and_hpo/job_payload",
        entrypoint="train_simple.py",
        compute_pool=COMPUTE_POOL,
        stage_name=STAGE,
        target_instances=1,
        env_vars={
            "DATABASE": DB,
            "SCHEMA": SCHEMA,
            "MODEL_NAME": "PATIENT_RISK_HPO_BEST",
            **best_params,
        },
    )

    print(f"Training best config as final model:")
    print(f"  Job ID: {job_best.id}")
    print(f"  Params: {best_params}")
else:
    print("No experiment runs found — run HPO first.")

## 5 | When to Use Ray Tune Instead

The Tuner API is the recommended starting point. However, for advanced use cases
you may want **Ray Tune** directly:

| Scenario | Use |
|----------|-----|
| Simple HPO, new project | **Tuner API** — 3 objects, automatic cluster |
| ASHA early stopping (kill bad trials early) | **Ray Tune** — scheduler support |
| Optuna / HyperOpt search algorithms | **Ray Tune** — broader algorithm library |
| Custom trial scheduling logic | **Ray Tune** — full control over trial lifecycle |
| Integration with existing Ray pipelines | **Ray Tune** — native Ray ecosystem |

### Ray Tune Resources

- [Snowflake Multi-Node ML Jobs](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/distributed-ml-jobs)
- [Ray Tune Documentation](https://docs.ray.io/en/latest/tune/index.html)
- [ASHA Scheduler](https://docs.ray.io/en/latest/tune/api/schedulers.html#asha)

With Ray Tune, you write a custom entrypoint that initializes a Ray cluster
across `target_instances` nodes, defines `ray.tune.Tuner`, and manages results
yourself. The trade-off is more code for more control.

## 6 | Summary

In this notebook we:

1. **Defined** a search space using `uniform`, `loguniform`, `randint`
2. **Configured** the Tuner with `RandomSearch` and 10 trials
3. **Submitted** the HPO job to SPCS via the Tuner API
4. **Analyzed** trial results and identified the best configuration
5. **Viewed** all trials in Experiment Tracking
6. **Registered** the best model in the Model Registry

### Key Takeaways

| Takeaway | Detail |
|----------|--------|
| **Tuner API** | High-level: `Tuner(func, space, config).run()` — clean and simple |
| **Experiment Tracking** | Each trial logged automatically for comparison |
| **Search space** | `loguniform` for learning rates, `randint` for tree counts |
| **Best model** | `results.best_model` ready for immediate registration |
| **Ray Tune** | Available for advanced scheduling (ASHA) and custom algorithms |

---

## Session 3 Complete!

```
 Session 3: Distributed Training & HPO
 ┌─────────────────────────────────────────────────────────────┐
 │                                                             │
 │  [NB 01] ML Jobs            [NB 02] Docker Images           │
 │  submit_directory()         Dockerfile + image registry     │
 │  env vars, logs, lifecycle  build, push, submit             │
 │                                                             │
 │  [NB 03] Distributed        [NB 04] HPO                    │
 │  XGBEstimator               Tuner API                       │
 │  DataConnector              search_space + TunerConfig       │
 │  Multi-node Ray cluster     Experiment Tracking per trial    │
 │                                                             │
 └─────────────────────────────────────────────────────────────┘
```

**Next session** → Session 4: Deployment, Real-Time Inference & Model Monitoring